# Teaching notebook: a model of visual search, trained from scratch

This notebook builds the priority-map model of visual search from
zero: we construct each piece, show what goes in and what comes out,
create data, and **train the model ourselves** with gradient descent.

**What the model does.** A person searches a display for a target
(say, a green diamond) while trying to ignore a bright red distractor.
The model predicts **which item their eyes go to**.

- **Input:** a picture of the display (pixels), the goal (a color and
  a shape to look for), where the eyes are now, and what happened on
  previous trials.
- **Output:** a probability for each item on the screen. One predicted
  eye movement = one random draw from those probabilities.

Run this from the `Visual-Search/` folder. Needs `numpy`,
`matplotlib`, `torch` (only for the training step).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from front_end import render, form_map, IMG
from build_contexts_v21 import (opponency_contrast, template_axis,
                                wedge_profiles, item_positions, NBINS, MAXR)

def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))

radii = np.linspace(0.09, MAXR, NBINS)   # distance bins along each ray
pos, _ = item_positions(6)               # the six item positions (a ring)

## The model at a glance

One picture before any code. Everything flows into a single
**priority map** — one number per item — and the eye movement is read
out from it.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 3.2))
boxes = [(0.04, "display\n(pixels)"),
         (0.20, "contrast\nmaps"),
         (0.36, "goal-weighted\nevidence"),
         (0.52, "x attention\nwindow"),
         (0.68, "+ memory of\npast trials"),
         (0.84, "priority map\n-> softmax")]
for x, label in boxes:
    ax.add_patch(plt.Rectangle((x, 0.35), 0.12, 0.32, fill=True,
                               fc="#eef2fa", ec="#334488", lw=1.5))
    ax.text(x + 0.06, 0.51, label, ha="center", va="center", fontsize=10)
for x, _ in boxes[:-1]:
    ax.annotate("", xy=(x + 0.16, 0.51), xytext=(x + 0.12, 0.51),
                arrowprops=dict(arrowstyle="->", lw=1.5))
ax.annotate("the goal:\n'find the green diamond'", xy=(0.42, 0.67),
            xytext=(0.36, 0.90), ha="center", fontsize=9,
            arrowprops=dict(arrowstyle="->", color="#118844"))
ax.annotate("where the eyes are now", xy=(0.58, 0.35),
            xytext=(0.52, 0.10), ha="center", fontsize=9,
            arrowprops=dict(arrowstyle="->", color="#884411"))
ax.annotate("what happened on\nearlier trials", xy=(0.74, 0.67),
            xytext=(0.72, 0.90), ha="center", fontsize=9,
            arrowprops=dict(arrowstyle="->", color="#aa2222"))
ax.text(0.99, 0.51, "next\nsaccade", ha="center", va="center", fontsize=10)
ax.annotate("", xy=(0.985, 0.51), xytext=(0.96, 0.51),
            arrowprops=dict(arrowstyle="->", lw=1.5))
ax.set_xlim(0, 1.05); ax.set_ylim(0, 1); ax.axis("off")
plt.title("the model: one priority map, three inputs into it")
plt.show()

## Step 1 — The input: a picture

We make one display: six items on a ring, all green, except one red
**distractor**. The **target** is the green diamond. The `+` is where
the eyes start.

The pixels are the model's entire stimulus input — no item is labeled
"target" or "distractor" anywhere.

In [ ]:
TARG, SING = 1, 4        # which slots hold the target and the distractor

items = []
for j in range(6):
    items.append(dict(x=pos[j][0], y=pos[j][1],
                      color="red" if j == SING else "green",
                      shape="diamond" if j == TARG else "circle"))
img = render(items)

plt.figure(figsize=(4, 4))
plt.imshow(img, origin="lower")
plt.plot(IMG / 2, IMG / 2, "k+", ms=12)
plt.title("the input: raw pixels (+ = fixation)")
plt.xticks([]); plt.yticks([])
plt.show()

## Step 2 — Early vision: contrast maps

The first stage mimics early visual cortex: at every location it asks
"how different is this spot from its surroundings?" — once for color
(red vs green) and once for brightness. This stage is **fixed** and
knows nothing about the task.

In [ ]:
maps = opponency_contrast(img)

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
v = np.abs(maps["RG"]).max()
axes[0].imshow(maps["RG"], origin="lower", cmap="RdBu_r", vmin=-v, vmax=v)
axes[0].set_title("color contrast (red +, green -)")
axes[1].imshow(maps["P"], origin="lower", cmap="magma")
axes[1].set_title("brightness contrast ('something is here')")
for ax in axes:
    ax.set_xticks([]); ax.set_yticks([])
plt.show()

## Step 3 — Sensing from the eyes, with the goal as a direction

Two ideas in one small function:

1. **The goal is a direction, not a label.** "Look for green" rotates
   the color axis so green scores positive and red scores negative.
   There is no "suppress red" rule — red just ends up on the wrong
   side of the axis.
2. **The world is sensed from the current fixation.** The maps are
   read along rays fanning out from the eyes (like a radar sweep), and
   an **attention window** weights near evidence more than far
   evidence.

The function below turns the picture into **three numbers per item**:
color evidence, "an object is there" evidence, and shape evidence.

In [ ]:
u = template_axis("green")     # the goal: a unit vector toward green

def item_evidence(items, img, maps, fixation, window_steepness=3.4,
                  window_reach=0.55):
    # read the maps along rays from the fixation -> radial profiles
    prof, _ = wedge_profiles(maps, u, fixation, 6)
    prof = prof / prof.std(axis=(0, 1), keepdims=True)   # common units
    # the attention window: near bins count more than far bins
    window = sigmoid(window_steepness * (window_reach - radii))
    color = (np.maximum(prof[:, :, 0], 0) * window).sum(axis=1)  # cut negatives
    presence = (np.maximum(prof[:, :, 2], 0) * window).sum(axis=1)
    # shape evidence: does this item match the goal shape? (one shortcut:
    # shape is read off a template map, not computed from pixels)
    fmap = form_map(items)
    px = lambda v: int((v + 0.75) / 1.5 * IMG)
    shape = np.array([fmap[px(pos[j][1]), px(pos[j][0])] for j in range(6)])
    shape = shape / shape.std()
    return color, presence, shape

color, presence, shape = item_evidence(items, img, maps, (0.0, 0.0))

x = np.arange(1, 7)
plt.figure(figsize=(7, 3.5))
plt.bar(x - 0.25, color, 0.25, label="color evidence")
plt.bar(x, presence, 0.25, label="object evidence")
plt.bar(x + 0.25, shape, 0.25, label="shape evidence")
plt.xlabel("item"); plt.legend()
plt.title(f"the picture, boiled down to 3 numbers per item\n"
          f"(item {TARG+1} = target, item {SING+1} = red distractor)")
plt.show()

Look at the bars: the red distractor (item 5) has almost no
*color* evidence — the "attend green" rotation already pushed it down
— while the target (item 2) alone has *shape* evidence.

One more observation: the *object* evidence is nearly the same for
every item (each slot holds exactly one object). Something that is
equal everywhere cannot help choose BETWEEN items — the softmax at the
end ignores it. So the model needs no weight for it, and from here on
we keep just two stimulus signals per item: **color** and **shape**.

## Step 4 — Memory of past trials

People are pulled toward where targets have recently been, and
slightly away from where distractors have been. The model keeps two
simple memories, one line each, updated after every trial:

    h = (1 - eta) * h        # everything fades a little
    h[location] += eta       # today's location gets a boost

`eta` sets the speed: big eta = fast learning, fast forgetting. We
simulate 30 trials where the distractor prefers one location:

In [ ]:
rng = np.random.default_rng(0)
eta_T, eta_D = 0.6, 0.2          # example speeds (we will LEARN these later)
hT, hD = np.zeros(6), np.zeros(6)
record = []
for t in range(30):
    targ = rng.integers(6)                                # target: anywhere
    sing = SING if rng.random() < 0.7 else rng.integers(6)  # biased
    hT = (1 - eta_T) * hT; hT[targ] += eta_T
    hD = (1 - eta_D) * hD; hD[sing] += eta_D
    record.append((hT.copy(), hD.copy()))

plt.figure(figsize=(7, 3.5))
plt.plot([r[0][SING] for r in record], label="target memory at slot 5")
plt.plot([r[1][SING] for r in record], label="distractor memory at slot 5")
plt.xlabel("trial"); plt.ylabel("memory strength"); plt.legend()
plt.title("two memories, two speeds: fast and jumpy vs slow and steady")
plt.show()

## Step 5 — The whole model in five lines

Everything meets in the priority map. Each item's priority is a
**weighted sum**: the three evidence numbers times three weights, plus
the two memories times two weights. Softmax turns priorities into
probabilities — **the output**.

The six numbers `w = (w_color, w_shape, w_memT, w_memD, eta_T,
eta_D)` are the model's ONLY unknowns. We have not chosen them yet —
below we try it **untrained**, with all weights equal.

In [ ]:
def predict(color, shape, hT, hD, w):
    priority = (w["color"] * color + w["shape"] * shape
                + w["memT"] * hT + w["memD"] * hD)
    p = np.exp(priority - priority.max())
    return p / p.sum()

untrained = dict(color=1, shape=1, memT=1, memD=1)
p = predict(color, shape, hT, hD, untrained)

plt.figure(figsize=(6, 3.2))
cols = ["#999999"] * 6; cols[TARG] = "#118844"; cols[SING] = "#aa2222"
plt.bar(x, 100 * p, color=cols)
plt.ylabel("P(saccade lands here) [%]"); plt.xlabel("item")
plt.title("the OUTPUT, before training: not much of a prediction yet")
plt.show()

## Step 6 — Make data to learn from

Real fitting used 217,595 eye movements from 333 people (not shipped
with this repo). Here we create a stand-in: a **simulated
participant** with known "true" weights, who does 2,000 trials. Because
we know the truth, we can check afterwards whether training recovers
it.

First we precompute the three evidence numbers for every possible
display (each target/distractor arrangement) — the slow part.

In [ ]:
TRUE = dict(color=1.2, shape=1.6, memT=2.0, memD=-1.2,
            eta_T=0.6, eta_D=0.2)

evidence = {}          # (target slot, distractor slot) -> (color, shape)
for tg in range(6):
    for sg in range(6):
        if sg == tg:
            continue
        its = [dict(x=pos[j][0], y=pos[j][1],
                    color="red" if j == sg else "green",
                    shape="diamond" if j == tg else "circle")
               for j in range(6)]
        im = render(its)
        c, o, sh = item_evidence(its, im, opponency_contrast(im), (0.0, 0.0))
        evidence[(tg, sg)] = [c, sh]
# put both signals in common units ACROSS all displays (one scale each)
allc = np.array([v[0] for v in evidence.values()])
alls = np.array([v[1] for v in evidence.values()])
for v in evidence.values():
    v[0] = v[0] / allc.std()
    v[1] = v[1] / alls.std()
print(f"prepared {len(evidence)} displays")

In [ ]:
n_trials = 2000
trials, choices = [], []
hT, hD = np.zeros(6), np.zeros(6)
FAVORITE = 4                       # the distractor's favorite location
for t in range(n_trials):
    tg = int(rng.integers(6))
    if rng.random() < 0.6 and tg != FAVORITE:
        sg = FAVORITE
    else:
        sg = int(rng.choice([j for j in range(6) if j != tg]))
    c, sh = evidence[(tg, sg)]
    w = TRUE
    p = predict(c, sh, hT, hD,
                dict(color=w["color"], shape=w["shape"],
                     memT=w["memT"], memD=w["memD"]))
    choice = int(rng.choice(6, p=p))          # the 'participant' looks somewhere
    trials.append((tg, sg)); choices.append(choice)
    hT = (1 - w["eta_T"]) * hT; hT[tg] += w["eta_T"]
    hD = (1 - w["eta_D"]) * hD; hD[sg] += w["eta_D"]
print("example data - trial 1:", trials[0], "-> looked at item", choices[0] + 1)

## Step 7 — Train from scratch

Now the actual training, exactly as in the real fits:

1. Start all weights at zero-ish values (the model knows nothing).
2. Replay the trials in order. The memory speeds `eta` are themselves
   trainable, so the memories are rebuilt inside the training loop.
3. The **loss** is how much probability the model failed to give to
   the item the participant actually looked at.
4. Gradient descent nudges all six numbers to make the loss smaller.
   Repeat a few hundred times. (This cell takes a few minutes — the
   memories are honestly rebuilt trial-by-trial on every pass.)

In [ ]:
import torch

C = torch.tensor(np.array([evidence[tr][0] for tr in trials]), dtype=torch.float32)
S = torch.tensor(np.array([evidence[tr][1] for tr in trials]), dtype=torch.float32)
targ_slots = torch.tensor([tr[0] for tr in trials])
sing_slots = torch.tensor([tr[1] for tr in trials])
looked = torch.tensor(choices)

w = {name: torch.tensor(0.1, requires_grad=True)
     for name in ["color", "shape", "memT", "memD"]}
raw_eta = {name: torch.tensor(0.0, requires_grad=True) for name in ["T", "D"]}
optimizer = torch.optim.Adam(list(w.values()) + list(raw_eta.values()), lr=0.05)

losses = []
for epoch in range(500):
    etaT = torch.sigmoid(raw_eta["T"]); etaD = torch.sigmoid(raw_eta["D"])
    hT = torch.zeros(6); hD = torch.zeros(6)
    HT, HD = [], []
    for t in range(n_trials):                 # rebuild memories, in order
        HT.append(hT); HD.append(hD)
        eT = torch.zeros(6); eT[targ_slots[t]] = 1.0
        eD = torch.zeros(6); eD[sing_slots[t]] = 1.0
        hT = (1 - etaT) * hT + etaT * eT
        hD = (1 - etaD) * hD + etaD * eD
    HT, HD = torch.stack(HT), torch.stack(HD)
    priority = (w["color"] * C + w["shape"] * S
                + w["memT"] * HT + w["memD"] * HD)
    loss = torch.nn.functional.cross_entropy(priority, looked)
    optimizer.zero_grad(); loss.backward(); optimizer.step()
    losses.append(loss.item())

plt.figure(figsize=(6, 3))
plt.plot(losses)
plt.xlabel("training step"); plt.ylabel("loss")
plt.title("training from scratch: the loss falls as the weights learn")
plt.show()

## Step 8 — Did it learn the right things?

Two checks. First, the learned weights against the true ones we used
to make the data. They come out close but not perfect — look at the
distractor-memory pair: the strength is too big and the speed too
small, in a way that nearly cancels. With 2,000 trials, some pairs of
settings trade off like this; the real fit, on 217,595 eye movements,
recovered every value within a few percent. Second check: the trained
model's prediction for our demo display — compare with the flat,
untrained bars from step 5.

In [ ]:
learned = {k: v.item() for k, v in w.items()}
learned["eta_T"] = torch.sigmoid(raw_eta["T"]).item()
learned["eta_D"] = torch.sigmoid(raw_eta["D"]).item()

names = list(TRUE)
xpos = np.arange(len(names))
plt.figure(figsize=(8, 3.5))
plt.bar(xpos - 0.18, [TRUE[n] for n in names], 0.36, label="true")
plt.bar(xpos + 0.18, [learned[n] for n in names], 0.36, label="learned")
plt.xticks(xpos, names); plt.axhline(0, color="k", lw=0.5); plt.legend()
plt.title("training roughly recovers the participant's settings")
plt.show()

demo_c, demo_s = evidence[(TARG, SING)]
p_trained = predict(demo_c, demo_s, hT.detach().numpy(),
                    hD.detach().numpy(),
                    {k: learned[k] for k in ["color", "shape",
                                             "memT", "memD"]})
plt.figure(figsize=(6, 3.2))
plt.bar(x, 100 * p_trained, color=cols)
plt.ylabel("P(saccade lands here) [%]"); plt.xlabel("item")
plt.title("the OUTPUT after training: target up, distractor down")
plt.show()

## Step 9 — A human effect appears by itself

Nothing in the model mentions "priming." But run the trained model
across many trials and count how often it looks at the distractor
when the distractor repeats its location vs moves: it looks **less**
on repeats — because the slow distractor memory is still parked
there. Real people show exactly this pattern.

In [ ]:
hT, hD = np.zeros(6), np.zeros(6)
prev_sing, repeat_p, change_p = None, [], []
ww = {k: learned[k] for k in ["color", "shape", "memT", "memD"]}
for t in range(2000):
    tg = int(rng.integers(6))
    sg = int(rng.choice([j for j in range(6) if j != tg]))
    c, sh = evidence[(tg, sg)]
    p = predict(c, sh, hT, hD, ww)
    (repeat_p if sg == prev_sing else change_p).append(p[sg])
    prev_sing = sg
    hT = (1 - learned["eta_T"]) * hT; hT[tg] += learned["eta_T"]
    hD = (1 - learned["eta_D"]) * hD; hD[sg] += learned["eta_D"]

plt.figure(figsize=(4.5, 3.5))
plt.bar(["distractor location\nREPEATED", "distractor location\nchanged"],
        [100 * np.mean(repeat_p), 100 * np.mean(change_p)],
        color=["#aa2222", "#dd9999"])
plt.ylabel("P(look at distractor) [%]")
plt.title("an effect nobody programmed: location priming")
plt.show()

## Recap

| The model receives | It computes | It returns |
| --- | --- | --- |
| a picture (pixels) | contrast maps -> goal-directed evidence, read from the fixation through an attention window | a **probability for each item** |
| a goal (color + shape) | the direction that evidence is measured along | (one saccade = one draw) |
| the trial history | two leaky memories, added with + and - signs | |

And training is nothing exotic: replay the trials, score the
probability given to each real eye movement, nudge six numbers to
make that score better.

The real model in this repository was trained the same way on
**217,595 eye movements from 333 people**. On people it never saw, it
gives the true choice 29% probability on average (chance: 18%) and
names the exact item first 54% of the time — and it reproduces the
classic findings of this literature (distractor suppression, its
persistence across saccades, and the priming effects). See
`RESULTS.md` for the numbers and
`docs/priority_field_visual_search_model.md` for the theory.